In [ ]:
from pathlib import Path
import os, sys, json, zipfile, shutil, subprocess, glob, math, time, random

PROJECT_ROOT = Path(os.environ.get("AIC_VQA_ROOT", Path.cwd())).resolve()
DEFAULT_PROJECT_ROOT = Path("/home/izu/Projects/AIC-UIT/adversarial-attack")
if not ((PROJECT_ROOT / "dataset").exists() and (PROJECT_ROOT / "Model").exists()) and DEFAULT_PROJECT_ROOT.exists():
    PROJECT_ROOT = DEFAULT_PROJECT_ROOT

DATA_DIR_OVERRIDE = os.environ.get("AIC_VQA_DATA_DIR", "").strip()
MODEL_DIR_OVERRIDE = os.environ.get("AIC_VQA_MODEL_DIR", "").strip()

WORK = PROJECT_ROOT
DATASET_DIR_HINT = Path(DATA_DIR_OVERRIDE).expanduser().resolve() if DATA_DIR_OVERRIDE else (WORK / "dataset")
MODEL_DIR_HINT = Path(MODEL_DIR_OVERRIDE).expanduser().resolve() if MODEL_DIR_OVERRIDE else (WORK / "Model")
EXTRACT_DIR = WORK / "extracted"
OUT_ROOT = WORK / "adv_outputs_smooth"
SUBMISSION_ZIP = WORK / "submission.zip"
TORCH_CACHE = WORK / "torch_cache"
os.environ["TORCH_HOME"] = str(TORCH_CACHE)

MAX_ZIP_MB = 49.20
MAX_GPU_WORKERS = 2

# FAST   = quick sanity
# COOK   = recommended
# INSANE = more steps/restarts near deadline
PRESET = "COOK"

if PRESET == "FAST":
    BATCH_IMAGES = 8
    ATTACK_STEPS = 45
    RESTARTS = 1
    LOW_RES = 112
    LR = 0.85 / 255.0
    TV_LAMBDA = 3.0
elif PRESET == "COOK":
    BATCH_IMAGES = 6
    ATTACK_STEPS = 75
    RESTARTS = 2
    LOW_RES = 112
    LR = 0.75 / 255.0
    TV_LAMBDA = 4.0
else:
    BATCH_IMAGES = 5
    ATTACK_STEPS = 120
    RESTARTS = 3
    LOW_RES = 112
    LR = 0.65 / 255.0
    TV_LAMBDA = 4.0

OPT_PSNR = 45.08
SAVE_MIN_PSNR = 45.02

L2_LAMBDA = 0.15
EVAL_EVERY = 5
SEED = 1337

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_DIR_HINT:", DATASET_DIR_HINT)
print("MODEL_DIR_HINT:", MODEL_DIR_HINT)
print("Preset:", PRESET, "| steps:", ATTACK_STEPS, "restarts:", RESTARTS, "low_res:", LOW_RES)


In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile, shutil, glob, json


def is_dataset_dir(p: Path) -> bool:
    return (p / "images").is_dir() and (p / "questions" / "test.json").exists()


def is_model_dir(p: Path) -> bool:
    needed = ["vit_model.py", "model.pth", "vocab.pth", "ans_vocab.pth"]
    return all((p / n).exists() for n in needed)


def find_named_file(candidates, roots):
    want = {c.lower() for c in candidates}
    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if p.is_file() and p.name.lower() in want:
                return p
    return None


def extract_zip_once(zip_path: Path, dest: Path):
    marker = dest / f".extracted_{zip_path.stem}"
    if marker.exists():
        print(f"Already extracted: {zip_path.name}")
        return
    print(f"Extracting {zip_path} -> {dest}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest)
    marker.write_text("ok")


EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Prefer already-downloaded local folders.
DATA_DIR = DATASET_DIR_HINT if is_dataset_dir(DATASET_DIR_HINT) else None
MODEL_DIR = MODEL_DIR_HINT if is_model_dir(MODEL_DIR_HINT) else None

# Fallback: if local folders are missing, look for dataset/model zip files in WORK.
if DATA_DIR is None or MODEL_DIR is None:
    dataset_zip = find_named_file(["dataset.zip"], [WORK])
    model_zip = find_named_file(["Model.zip", "model.zip"], [WORK])

    print("dataset_zip:", dataset_zip)
    print("model_zip:", model_zip)

    if DATA_DIR is None and dataset_zip is not None:
        extract_zip_once(dataset_zip, EXTRACT_DIR)
    if MODEL_DIR is None and model_zip is not None:
        extract_zip_once(model_zip, EXTRACT_DIR)


def find_dataset_dir():
    roots = [DATASET_DIR_HINT, EXTRACT_DIR, WORK]
    for root in roots:
        if not root.exists():
            continue
        for p in [root] + [x for x in root.rglob("*") if x.is_dir()]:
            if is_dataset_dir(p):
                return p
    raise FileNotFoundError(
        "Could not find dataset dir. Expected images/ and questions/test.json under dataset/."
    )


def find_model_dir():
    roots = [MODEL_DIR_HINT, EXTRACT_DIR, WORK]
    for root in roots:
        if not root.exists():
            continue
        for p in [root] + [x for x in root.rglob("*") if x.is_dir()]:
            if is_model_dir(p):
                return p
    raise FileNotFoundError(
        "Could not find model dir. Expected vit_model.py, model.pth, vocab.pth, ans_vocab.pth under Model/."
    )

if DATA_DIR is None:
    DATA_DIR = find_dataset_dir()
if MODEL_DIR is None:
    MODEL_DIR = find_model_dir()

print("DATA_DIR:", DATA_DIR)
print("MODEL_DIR:", MODEL_DIR)

with open(DATA_DIR / "questions" / "test.json", "r") as f:
    q_data = json.load(f)["questions"]

unique_files = sorted({x["image_filename"] for x in q_data})
print("Questions:", len(q_data), "| unique images:", len(unique_files))
print("First item:", q_data[0])


In [ ]:
import torch
print("torch:", torch.__version__)

try:
    import torchvision.models as tvm
    print("Pre-caching torchvision ViT_B_16_Weights.DEFAULT...")
    tmp_vit = tvm.vit_b_16(weights=tvm.ViT_B_16_Weights.DEFAULT)
    del tmp_vit
    print("ViT weights cached under:", os.environ.get("TORCH_HOME"))
except Exception as e:
    print("Could not pre-cache ViT weights:", repr(e))
    print("Turn Kaggle Internet ON, or attach/cache the torchvision ViT-B/16 weights.")


In [ ]:
from pathlib import Path
from collections import OrderedDict
import os, sys, json, random, math, time, shutil, zipfile, subprocess

import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F


def torch_load(path, map_location=None):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)


def clean_state_dict(sd):
    if isinstance(sd, dict) and "state_dict" in sd and isinstance(sd["state_dict"], dict):
        sd = sd["state_dict"]
    if isinstance(sd, dict) and all(str(k).startswith("module.") for k in sd.keys()):
        sd = {str(k)[7:]: v for k, v in sd.items()}
    return sd


def encode_question(q, vocab, max_len):
    tokens = q.lower().replace("?", "").split()
    ids = [vocab.get(w, 1) for w in tokens]
    ids = ids[:max_len]
    ids += [0] * (max_len - len(ids))
    return torch.tensor(ids, dtype=torch.long)


def load_u8(path):
    arr = np.array(Image.open(path).convert("RGB"), dtype=np.uint8)
    return torch.from_numpy(arr).permute(2, 0, 1).contiguous()


def u8_to_float(x):
    return x.float() / 255.0


def save_png_max(path, x_u8):
    arr = x_u8.permute(1, 2, 0).detach().cpu().numpy().astype(np.uint8)
    Image.fromarray(arr, mode="RGB").save(path, format="PNG", optimize=True, compress_level=9)


def psnr_u8(clean_u8, adv_u8):
    mse = (clean_u8.float() - adv_u8.float()).pow(2).flatten(1).mean(1)
    return 10.0 * torch.log10((255.0 * 255.0) / (mse + 1e-12))


def project_psnr_l2(x0, x, psnr):
    # x0/x are floats in [0, 1]. For MAX=1, PSNR = -10*log10(MSE).
    max_mse = 10.0 ** (-psnr / 10.0)
    delta = x - x0
    mse = delta.pow(2).flatten(1).mean(1).view(-1, 1, 1, 1)
    scale = torch.sqrt(torch.tensor(max_mse, device=x.device, dtype=x.dtype) / (mse + 1e-12))
    scale = torch.clamp(scale, max=1.0)
    return torch.clamp(x0 + delta * scale, 0.0, 1.0)


def quantize_with_psnr_safety(x0, adv, min_psnr):
    clean_u8 = torch.round(x0 * 255.0).clamp(0, 255).to(torch.uint8)
    scale = torch.ones((x0.size(0), 1, 1, 1), device=x0.device, dtype=x0.dtype)

    # Binary-ish repeated shrink: survives uint8 rounding.
    for _ in range(45):
        cand = torch.clamp(x0 + (adv - x0) * scale, 0.0, 1.0)
        adv_u8 = torch.round(cand * 255.0).clamp(0, 255).to(torch.uint8)
        p = psnr_u8(clean_u8, adv_u8)
        bad = p < min_psnr
        if not bool(bad.any()):
            return adv_u8, p
        scale[bad.view(-1, 1, 1, 1)] *= 0.97

    cand = torch.clamp(x0 + (adv - x0) * scale, 0.0, 1.0)
    adv_u8 = torch.round(cand * 255.0).clamp(0, 255).to(torch.uint8)
    return adv_u8, psnr_u8(clean_u8, adv_u8)


def tv_loss(x):
    # Isotropic-ish TV, intentionally simple and stable.
    dh = (x[:, :, 1:, :] - x[:, :, :-1, :]).abs().mean()
    dw = (x[:, :, :, 1:] - x[:, :, :, :-1]).abs().mean()
    return dh + dw


class ModelWrapper:
    def __init__(self, model_dir, device):
        self.device = device
        model_dir = Path(model_dir)
        sys.path.insert(0, str(model_dir))
        from vit_model import ViTModel, Config

        self.Config = Config
        self.vocab = torch_load(model_dir / "vocab.pth", map_location="cpu")
        self.ans_vocab = torch_load(model_dir / "ans_vocab.pth", map_location="cpu")

        self.model = ViTModel(
            vocab_size=len(self.vocab),
            num_answer=len(self.ans_vocab),
            d_model=Config.D_MODEL,
            n_heads=Config.N_HEADS,
            n_layers=Config.N_LAYERS,
            ff_dim=Config.FF_DIM,
            dropout=Config.DROPOUT,
            max_q_len=Config.MAX_LEN,
        ).to(device)

        sd = clean_state_dict(torch_load(model_dir / "model.pth", map_location=device))
        missing, unexpected = self.model.load_state_dict(sd, strict=False)
        print(f"[model] missing={len(missing)} unexpected={len(unexpected)}", flush=True)

        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

        self.mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    def preprocess(self, x):
        if x.shape[-2:] != (224, 224):
            x = F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False)
        return (x - self.mean) / self.std

    def logits(self, raw_float, q_tokens):
        return self.model(self.preprocess(raw_float), q_tokens)


def grouped_objective(logits, y, map_idx, batch_images, tau=0.35):
    n = logits.size(0)
    ar = torch.arange(n, device=logits.device)

    correct = logits[ar, y]
    masked = logits.clone()
    masked[ar, y] = -1e9
    other = masked.max(1).values
    margin = other - correct

    ce = F.cross_entropy(logits, y, reduction="none")

    # Force every question attached to the same image to move, not only the easy ones.
    softmins = []
    for b in range(batch_images):
        mb = margin[map_idx == b]
        if mb.numel():
            softmins.append(-tau * torch.logsumexp(-mb / tau, dim=0))
    softmin = torch.stack(softmins).mean() if softmins else margin.mean()

    # CE helps leave the clean answer; margin/softmin push the top non-clean answer over it.
    return ce.mean() + 0.70 * margin.mean() + 0.85 * softmin


@torch.no_grad()
def predict(wrapper, x_float, q_tokens, map_idx, batch_images, chunk=48):
    preds = []
    for s in range(0, q_tokens.size(0), chunk):
        q = q_tokens[s:s + chunk]
        mi = map_idx[s:s + chunk]
        logits = wrapper.logits(x_float[mi], q)
        preds.append(logits.argmax(1))
    return torch.cat(preds, dim=0)


@torch.no_grad()
def score_candidate(wrapper, adv, q_tokens, map_idx, clean_y, batch_images):
    logits = wrapper.logits(adv[map_idx], q_tokens)
    pred = logits.argmax(1)

    n = logits.size(0)
    ar = torch.arange(n, device=logits.device)
    correct = logits[ar, clean_y]
    masked = logits.clone()
    masked[ar, clean_y] = -1e9
    other = masked.max(1).values
    margin = other - correct

    scores = []
    succ_rates = []
    for b in range(batch_images):
        m = map_idx == b
        if m.any():
            sr = (pred[m] != clean_y[m]).float().mean()
            # Success dominates; margin breaks ties and helps duplicated-question images.
            sc = 100.0 * sr + margin[m].mean() + 0.75 * margin[m].min()
            scores.append(sc)
            succ_rates.append(sr)
        else:
            scores.append(torch.tensor(-1e9, device=adv.device))
            succ_rates.append(torch.tensor(0.0, device=adv.device))
    return torch.stack(scores), torch.stack(succ_rates), pred


def make_adv_from_lowres(x0, z_low, psnr):
    delta = F.interpolate(z_low, size=x0.shape[-2:], mode="bicubic", align_corners=False)
    x = torch.clamp(x0 + delta, 0.0, 1.0)
    x = project_psnr_l2(x0, x, psnr)
    delta = x - x0
    return x, delta


def attack_batch(wrapper, x0, q_tokens, map_idx, clean_y, cfg):
    device = x0.device
    B = x0.size(0)
    best_adv = x0.clone()
    best_score = torch.full((B,), -1e9, device=device)

    for r in range(cfg["restarts"]):
        if r == 0:
            init = torch.zeros((B, 3, cfg["low_res"], cfg["low_res"]), device=device)
        else:
            init = torch.empty((B, 3, cfg["low_res"], cfg["low_res"]), device=device).uniform_(-1.25/255.0, 1.25/255.0)

        z = init.detach().clone().requires_grad_(True)
        opt = torch.optim.Adam([z], lr=cfg["lr"], betas=(0.85, 0.999), eps=1e-8)

        for step in range(cfg["steps"]):
            opt.zero_grad(set_to_none=True)
            adv, delta = make_adv_from_lowres(x0, z, cfg["opt_psnr"])

            logits = wrapper.logits(adv[map_idx], q_tokens)
            attack_obj = grouped_objective(logits, clean_y, map_idx, B)

            smooth = tv_loss(delta)
            l2 = delta.pow(2).mean()
            loss = -attack_obj + cfg["tv_lambda"] * smooth + cfg["l2_lambda"] * l2
            loss.backward()

            # Keep optimizer stable if a bad gradient appears.
            if z.grad is not None:
                z.grad.data = torch.nan_to_num(z.grad.data, nan=0.0, posinf=0.0, neginf=0.0)

            opt.step()

            # z is only a low-res parameter; PSNR projection happens after upsampling.
            # Clipping prevents Adam from wasting steps on huge values that project to the same image.
            with torch.no_grad():
                z.clamp_(-8.0/255.0, 8.0/255.0)

            if ((step + 1) % cfg["eval_every"] == 0) or (step + 1 == cfg["steps"]):
                with torch.no_grad():
                    cand, _ = make_adv_from_lowres(x0, z, cfg["opt_psnr"])
                    scores, _, _ = score_candidate(wrapper, cand, q_tokens, map_idx, clean_y, B)
                    better = scores > best_score
                    if bool(better.any()):
                        best_adv[better] = cand.detach()[better]
                        best_score[better] = scores[better]

    return best_adv.detach()


def build_groups(data_dir):
    with open(Path(data_dir) / "questions" / "test.json", "r") as f:
        questions = json.load(f)["questions"]
    groups = OrderedDict()
    for item in questions:
        groups.setdefault(item["image_filename"], []).append(item)
    return list(groups.items())


def run_attack_rank(rank, world, gpu, data_dir, model_dir, out_root, cfg):
    import torch
    from pathlib import Path

    random.seed(cfg["seed"] + rank)
    np.random.seed(cfg["seed"] + rank)
    torch.manual_seed(cfg["seed"] + rank)

    if torch.cuda.is_available():
        torch.cuda.set_device(gpu)
        device = torch.device(f"cuda:{gpu}")
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    else:
        device = torch.device("cpu")

    out_dir = Path(out_root) / f"gpu{rank}"
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"[rank {rank}] device={device} out={out_dir}", flush=True)

    wrapper = ModelWrapper(model_dir, device)
    max_len = wrapper.Config.MAX_LEN

    all_groups = build_groups(data_dir)
    groups = all_groups[rank::world]
    print(f"[rank {rank}] groups={len(groups)} / total={len(all_groups)}", flush=True)

    total_q = 0
    success_q = 0
    psnr_values = []
    image_count = 0
    per_image = []

    pbar = tqdm(range(0, len(groups), cfg["batch_images"]), desc=f"rank{rank}/gpu{gpu}", position=rank)
    for start in pbar:
        batch_groups = groups[start:start + cfg["batch_images"]]
        B = len(batch_groups)

        clean_u8_list = []
        q_list = []
        map_list = []
        filename_list = []
        q_count_per_img = []

        for bi, (fn, items) in enumerate(batch_groups):
            filename_list.append(fn)
            clean_u8 = load_u8(Path(data_dir) / "images" / fn)
            clean_u8_list.append(clean_u8)
            q_count_per_img.append(len(items))

            for item in items:
                q_list.append(encode_question(item["question"], wrapper.vocab, max_len))
                map_list.append(bi)

        clean_u8 = torch.stack(clean_u8_list, dim=0).to(device, non_blocking=True)
        x0 = u8_to_float(clean_u8)
        q_tokens = torch.stack(q_list, dim=0).to(device, non_blocking=True)
        map_idx = torch.tensor(map_list, dtype=torch.long, device=device)

        with torch.no_grad():
            clean_y = predict(wrapper, x0, q_tokens, map_idx, B)

        adv = attack_batch(wrapper, x0, q_tokens, map_idx, clean_y, cfg)
        adv_u8, psnrs = quantize_with_psnr_safety(x0, adv, cfg["save_min_psnr"])
        adv_float_saved = u8_to_float(adv_u8).to(device)

        with torch.no_grad():
            adv_pred = predict(wrapper, adv_float_saved, q_tokens, map_idx, B)
            success = adv_pred != clean_y

        offset = 0
        for bi, fn in enumerate(filename_list):
            save_png_max(out_dir / fn, adv_u8[bi].detach().cpu())

            k = q_count_per_img[bi]
            succ_i = int(success[offset:offset + k].sum().item())
            total_i = int(k)
            offset += k

            ps = float(psnrs[bi].detach().cpu().item())
            total_q += total_i
            success_q += succ_i
            psnr_values.append(ps)
            image_count += 1
            per_image.append({"file": fn, "success": succ_i, "total": total_i, "psnr": ps})

        pbar.set_postfix({
            "ASR": f"{100.0 * success_q / max(total_q, 1):.1f}%",
            "minPSNR": f"{min(psnr_values):.2f}",
        })

        del clean_u8, x0, q_tokens, map_idx, clean_y, adv, adv_u8, adv_float_saved, adv_pred
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    stats = {
        "rank": rank,
        "gpu": gpu,
        "images": image_count,
        "questions": total_q,
        "success": success_q,
        "asr": success_q / max(total_q, 1),
        "min_psnr": min(psnr_values) if psnr_values else None,
        "mean_psnr": sum(psnr_values) / max(len(psnr_values), 1),
        "per_image": per_image,
    }
    with open(out_dir / f"stats_rank{rank}.json", "w") as f:
        json.dump(stats, f, indent=2)

    print(f"[rank {rank}] done ASR={100.0 * stats['asr']:.2f}% minPSNR={stats['min_psnr']}", flush=True)


In [ ]:
import os, subprocess, multiprocessing as mp, shutil, json, time
from pathlib import Path

def detect_gpu_count_no_torch_cuda():
    vis = os.environ.get("CUDA_VISIBLE_DEVICES")
    if vis and vis.strip() and vis.strip() != "-1":
        parts = [x for x in vis.split(",") if x.strip()]
        if parts:
            return len(parts)
    try:
        out = subprocess.check_output(["nvidia-smi", "-L"], text=True, stderr=subprocess.DEVNULL)
        return len([ln for ln in out.splitlines() if ln.strip().startswith("GPU ")])
    except Exception:
        return 0

gpu_count = detect_gpu_count_no_torch_cuda()
WORLD = max(1, min(MAX_GPU_WORKERS, gpu_count if gpu_count > 0 else 1))
print("Detected GPUs:", gpu_count, "| workers:", WORLD)

cfg = {
    "batch_images": BATCH_IMAGES,
    "steps": ATTACK_STEPS,
    "restarts": RESTARTS,
    "low_res": LOW_RES,
    "lr": LR,
    "tv_lambda": TV_LAMBDA,
    "l2_lambda": L2_LAMBDA,
    "eval_every": EVAL_EVERY,
    "opt_psnr": OPT_PSNR,
    "save_min_psnr": SAVE_MIN_PSNR,
    "seed": SEED,
}

OUT_ROOT.mkdir(parents=True, exist_ok=True)
for p in OUT_ROOT.glob("gpu*"):
    if p.is_dir():
        shutil.rmtree(p)

# Use fork so functions defined in this notebook are available to child processes.
# Parent avoided CUDA initialization above, so children own their CUDA contexts.
ctx = mp.get_context("fork")
procs = []
start_time = time.time()

for rank in range(WORLD):
    gpu = rank if gpu_count > 0 else 0
    p = ctx.Process(
        target=run_attack_rank,
        args=(rank, WORLD, gpu, str(DATA_DIR), str(MODEL_DIR), str(OUT_ROOT), cfg),
    )
    p.start()
    procs.append(p)

failed = []
for rank, p in enumerate(procs):
    p.join()
    print(f"rank {rank} exitcode:", p.exitcode)
    if p.exitcode != 0:
        failed.append((rank, p.exitcode))

if failed:
    raise RuntimeError(f"Some attack workers failed: {failed}")

print(f"Attack finished in {(time.time() - start_time)/60:.1f} min")


In [ ]:
from pathlib import Path
import os, json, zipfile, shutil
import numpy as np
from PIL import Image

def make_zip_from_pngs(zip_path, png_map, expected_files):
    zip_path = Path(zip_path)
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as zf:
        for name in expected_files:
            zf.write(png_map[name], arcname=name)
    return zip_path.stat().st_size / (1024 * 1024)

def collect_outputs(out_root):
    pngs = {}
    for shard in sorted(Path(out_root).glob("gpu*")):
        for p in shard.glob("*.png"):
            pngs[p.name] = p
    return pngs

with open(DATA_DIR / "questions" / "test.json", "r") as f:
    q_data = json.load(f)["questions"]
expected_files = sorted({x["image_filename"] for x in q_data})

pngs = collect_outputs(OUT_ROOT)
missing = sorted(set(expected_files) - set(pngs))
extra = sorted(set(pngs) - set(expected_files))

print("Expected images:", len(expected_files))
print("Generated images:", len(pngs))
print("Missing:", len(missing), missing[:5])
print("Extra:", len(extra), extra[:5])
assert not missing, "Missing required adversarial images."

size_mb = make_zip_from_pngs(SUBMISSION_ZIP, pngs, expected_files)
print(f"Initial submission size: {size_mb:.2f} MB")

# If still above the cap, progressively shrink perturbations toward clean images.
# This is only a last resort; low-res + TV usually keeps the zip below 50 MB already.
if size_mb > MAX_ZIP_MB:
    print(f"Zip is above {MAX_ZIP_MB:.2f} MB, starting size guard shrink...")
    SHRUNK_ROOT = WORK / "adv_outputs_size_guard"
    if SHRUNK_ROOT.exists():
        shutil.rmtree(SHRUNK_ROOT)
    SHRUNK_ROOT.mkdir(parents=True, exist_ok=True)

    best_size = size_mb
    best_factor = 1.0
    best_map = pngs

    for factor in [0.96, 0.93, 0.90, 0.87, 0.84, 0.80, 0.76, 0.72, 0.68]:
        trial_dir = SHRUNK_ROOT / f"factor_{factor:.2f}"
        trial_dir.mkdir(parents=True, exist_ok=True)
        trial_map = {}

        min_psnr = 999.0
        for name in expected_files:
            clean = np.array(Image.open(DATA_DIR / "images" / name).convert("RGB"), dtype=np.float32)
            adv = np.array(Image.open(pngs[name]).convert("RGB"), dtype=np.float32)

            shrunk = np.rint(clean + (adv - clean) * factor).clip(0, 255).astype(np.uint8)

            mse = np.mean((clean - shrunk.astype(np.float32)) ** 2)
            psnr = 10.0 * np.log10((255.0 * 255.0) / max(mse, 1e-12))
            min_psnr = min(min_psnr, float(psnr))

            out = trial_dir / name
            Image.fromarray(shrunk, mode="RGB").save(out, format="PNG", optimize=True, compress_level=9)
            trial_map[name] = out

        trial_zip = WORK / f"submission_factor_{factor:.2f}.zip"
        trial_size = make_zip_from_pngs(trial_zip, trial_map, expected_files)
        print(f"factor={factor:.2f} size={trial_size:.2f} MB minPSNR={min_psnr:.2f}")

        if trial_size < best_size:
            best_size = trial_size
            best_factor = factor
            best_map = trial_map

        if trial_size <= MAX_ZIP_MB and min_psnr >= SAVE_MIN_PSNR:
            shutil.copy2(trial_zip, SUBMISSION_ZIP)
            pngs = trial_map
            size_mb = trial_size
            print(f"Using factor={factor:.2f}")
            break

    if size_mb > MAX_ZIP_MB:
        print(f"WARNING: best shrink factor={best_factor:.2f} produced {best_size:.2f} MB, still above cap.")
        print("Try LOW_RES=56, PRESET='FAST', or lower ATTACK_STEPS slightly and rerun.")
    else:
        print(f"Final guarded submission size: {size_mb:.2f} MB")

# Summarize worker stats.
stats_files = sorted(Path(OUT_ROOT).glob("gpu*/stats_rank*.json"))
total_q = total_success = total_images = 0
min_psnr = 999.0
for sf in stats_files:
    st = json.loads(sf.read_text())
    print(sf.name, "ASR:", f"{100*st['asr']:.2f}%", "images:", st["images"], "minPSNR:", st["min_psnr"])
    total_q += st["questions"]
    total_success += st["success"]
    total_images += st["images"]
    if st["min_psnr"] is not None:
        min_psnr = min(min_psnr, st["min_psnr"])

final_size = SUBMISSION_ZIP.stat().st_size / (1024 * 1024)
print("=" * 60)
print("Final submission:", SUBMISSION_ZIP)
print(f"Final size: {final_size:.2f} MB")
print("Images:", total_images)
print("Local attack success vs clean predictions:", f"{100*total_success/max(total_q,1):.2f}%")
print("Local min PSNR before any size-guard shrink:", f"{min_psnr:.2f}")
print("=" * 60)

assert final_size <= 50.0, "Website cap is 50 MB; lower LOW_RES or rerun with stronger smoothing."
